### Embedding Models

Here we play around with some local embedding models to learn how semantic search happens in RAG systems. 

In [10]:
import pdfplumber

def extract_text(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        return " ".join(page.extract_text() for page in pdf.pages)

"""
It helps to give the documents to the models chunk by chunk 
since they have a limited context window. It helps with the performance. 
"""
def chunk_text(text, chunk_size=500):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

raw_text = extract_text("../docs/paper1.pdf")
chunks = chunk_text(raw_text)

In [11]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks, show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [13]:
query = "Model"
query_vec = model.encode([query])

top_k = 3
distances, indices = index.search(np.array(query_vec), top_k)

for i, idx in enumerate(indices[0]):
    print(f"Match {i+1}:\n{chunks[idx]}\n")

Match 1:
71.6 67.2 56.3 22.7 45.6 59.5 71.6 55.3 Table9: PerclassCIFAR-10classificationaccuracy. Classes aero car bird cat deer dog frog horse ship truck Supervised 93.7 96.3 89.4 82.4 93.6 89.7 95.0 94.3 95.7 95.2 (Ours)RotNet 91.7 95.8 87.1 83.5 91.5 85.3 94.2 91.9 95.7 94.2 16

Match 2:
17.5 RandomrescaledKra¨henbu¨hletal.(2015) 21.4 26.2 27.1 26.1 24.0 Context(Doerschetal.,2015) 19.7 26.7 31.9 32.7 30.9 ContextEncoders(Pathaketal.,2016b) 18.2 23.2 23.4 21.9 18.4 Colorization(Zhangetal.,2016a) 16.0 25.7 29.6 30.3 29.7 JigsawPuzzles(Noroozi&Favaro,2016) 23.0 31.9 35.0 34.2 29.3 BIGAN(Donahueetal.,2016) 22.0 28.7 31.8 31.3 29.7 Split-Brain(Zhangetal.,2016b) 21.3 30.7 34.0 34.1 32.5 Counting(Noroozietal.,2017) 23.3 33.9 36.3 34.7 29.6 (Ours)RotNet 21.5 31.0 35.1 34.6 33.7 classificationtasksofImageNet,Places,andPASCALVOCdatasetsandontheobjectdetectionand objectsegmentationtasksofPASCALVOC. Implementation details: For those experiments we implemented our RotNet model with an AlexNet arc